In [4]:
import os
import pandas as pd

data_folder = "./processed_data"
files = os.listdir(data_folder)
butterfly_abundance_raw_processed = pd.read_excel(os.path.join(data_folder, "butterfly_abundance.xlsx"), sheet_name=None)
connectivity_raw_processed = pd.read_excel(os.path.join(data_folder, "connectivity.xlsx"), sheet_name=None)
plant_abundance_raw_processed = pd.read_excel(os.path.join(data_folder, "plant_abundance.xlsx"), sheet_name=None)
pollinating_insects_raw_processed = pd.read_excel(os.path.join(data_folder, "pollinating_insects.xlsx"), sheet_name=None)

In [11]:
# Find the time range
# extract dates from all dataframes using regex

all_dates = set()
earliest_date = float('inf')
latest_date = float('-inf')

for sheet in connectivity_raw_processed.values():
    all_dates.update(sheet["Year"])

for sheet in butterfly_abundance_raw_processed.values():
    all_dates.update(sheet["Year"])

for sheet in plant_abundance_raw_processed.values():
    all_dates.update(sheet["Year"])

for sheet in pollinating_insects_raw_processed.values():
    all_dates.update(sheet["Year"])

earliest_date = min(all_dates)
latest_date = max(all_dates)

In [8]:
print(f"Time range: {earliest_date} - {latest_date}")

Time range: 1976 - 2024


In [10]:
# Add columns for habitat connectivity
habitat_connectivity_columns = [
    "butterfly_connectivity_composite_unsmoothed_index",
    "birds_connectivity_composite_unsmoothed_index",
    "mean_connectivity_woodland",
    "mean_connectivity_grassland",
    # The mean connectivity value is a measure of the relative connectivity on a scale of 0 to 100. 
    # Typical values are less than 1. 
    # There was a significant change in mean connectivity on neutral grassland between 1990 and 2007.
]

# Add columns for plants in the wider countryside
plant_abundance_columns = [
    "plant_abundance_arable_unsmoothed_index",
    "plant_abundance_bog_and_heath_unsmoothed_index",
    "plant_abundance_grassland_unsmoothed_index",
    "plant_abundance_woodland_unsmoothed_index",
]
# Add columns for insects in the wider countryside
butterfly_abundance_columns = [
    "butterfly_abundance_all_species_unsmoothed_index",
    "butterfly_abundance_all_specialist_species_unsmoothed_index",
    "butterfly_abundance_all_generalist_species_unsmoothed_index",
    "butterfly_abundance_farmland_all_species_unsmoothed_index",
    "butterfly_abundance_farmland_specialist_species_unsmoothed_index",
    "butterfly_abundance_farmland_generalist_species_unsmoothed_index",
    "butterfly_abundance_woodland_all_species_unsmoothed_index",
    "butterfly_abundance_woodland_specialist_species_unsmoothed_index",
    "butterfly_abundance_woodland_generalist_species_unsmoothed_index",
]

pollinator_columns = [
    # Add columns for pollinating insects
    "pollinators_mean_occupancy",
    "pollinating_wild_bees_mean_occupancy",
    "pollinating_hoverflies_mean_occupancy",
]

In [64]:
index = range(earliest_date, latest_date + 1)

# Create individual dataframes for each category then combine
def create_df_from_raw_processed(raw_processed, columns):
    df = pd.DataFrame(index=index, columns=columns)
    for i, sheet in enumerate(raw_processed.values()):
        temp = sheet.set_index("Year").iloc[:,0]
        temp = temp.reindex(df.index)
        temp = pd.to_numeric(temp, errors='coerce')

        curr_col = columns[i]
        df[curr_col] = temp
    return df

connectivity_df = create_df_from_raw_processed(connectivity_raw_processed, habitat_connectivity_columns)
butterfly_abundance_df = create_df_from_raw_processed(butterfly_abundance_raw_processed, butterfly_abundance_columns)
plant_abundance_df = create_df_from_raw_processed(plant_abundance_raw_processed, plant_abundance_columns)
pollinating_insects_df = create_df_from_raw_processed(pollinating_insects_raw_processed, pollinator_columns)



In [67]:
combined_data = pd.concat([connectivity_df, butterfly_abundance_df, plant_abundance_df, pollinating_insects_df], axis=1)

In [69]:
combined_data.to_excel(os.path.join(data_folder, "combined_data.xlsx"), index_label="Year")